# Monthly National-Series CUSUM and Lead-Lag Analysis
#### Hoss Ariannejad - GP132 - March'26

## Introduction
The notebook has two purposes:

1. Detect structural breaks in transformed monthly national signals using CUSUM.
2. Quantify the full-sample lead-lag relationship between monthly WTI price and the national activity series `rig_count` and `us_production_mbbld`.


## Scope

The analysis is restricted to national monthly series from `master_monthly.csv`:

- `wti_price_weekly`: WTI price aggregated to monthly frequency
- `rig_count`: national rotary rig count
- `us_production_mbbld`: U.S. crude oil production
- `wti_mom_pct`: month-over-month change in WTI
- `rig_count_diff`: month-over-month change in rig count
- `us_production_mbbld_diff`: month-over-month change in production

## Analytical workflow

The notebook is broken into the following sections:

- Data loading and monthly-series preparation
- CUSUM break detection on transformed national signals
- Lead-lag estimation on monthly national levels using cross-correlation and lagged OLS
- Full-sample interpretation and summary findings

## Lag convention

Positive lag `k > 0` is defined as comparing `WTI_t` with `target_(t+k)`. Under this convention, a positive peak lag means WTI leads the target by `k` months.

In [31]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go

## Data Inputs and Preparation

The source data for this notebook is the processed monthly master table produced by the project pipeline. The table is sorted by date and extended with two simple first-difference series used in the CUSUM section.

The transformed series are used only for break detection. The lead-lag section uses the monthly level series because the project's lag-analysis outputs and dashboard views are defined on monthly levels.

In [14]:
NOTEBOOK_DIR = Path.cwd()
DATA_PATH = (NOTEBOOK_DIR / ".." / "grp132_datawrangler" / "data" / "processed" / "master_monthly.csv").resolve()

monthly = pd.read_csv(DATA_PATH, parse_dates=["date"]).sort_values("date")
monthly["rig_count_diff"] = monthly["rig_count"].diff()
monthly["us_production_mbbld_diff"] = monthly["us_production_mbbld"].diff()

monthly[[
    "date",
    "wti_price_weekly",
    "rig_count",
    "us_production_mbbld",
    "wti_mom_pct",
    "rig_count_diff",
    "us_production_mbbld_diff",
]].head()

,date,wti_price_weekly,rig_count,us_production_mbbld,wti_mom_pct,rig_count_diff,us_production_mbbld_diff
0,2013-01-01,94.0350,2259.00,7083.0,7.134923,NaN,NaN
1,2013-02-01,96.2100,2403.75,7145.0,2.312969,144.75,62.0
2,2013-03-01,93.0040,2220.00,7214.0,-3.332294,-183.75,69.0
3,2013-04-01,91.8575,1907.75,7374.0,-1.232743,-312.25,160.0
4,2013-05-01,94.3940,1894.60,7326.0,2.761342,-13.15,-48.0


## Part 1: CUSUM Break Detection

CUSUM is used here to identify major regime changes in the transformed national signals. The transformed signals were selected because they emphasize oil-price shocks and operational responses rather than long-run level drift.

### Inputs to the CUSUM section

- `wti_mom_pct`: price shock signal
- `rig_count_diff`: drilling response signal
- `us_production_mbbld_diff`: production response signal

### Method

For a standardized series `z_t`, the resetting CUSUM statistics are:

- `S+_t = max(0, S+_(t-1) + z_t - k)`
- `S-_t = max(0, S-_(t-1) - z_t - k)`

A break is recorded when either cumulative statistic exceeds the detection threshold. After detection, both statistics are reset to zero so the algorithm can find multiple break episodes over the sample.

### Parameters

- `threshold = 5.0`
- `drift = 0.5`

These settings match the defaults used in the dashboard CUSUM implementation.

In [44]:
def cusum_detect(series: pd.Series, threshold: float = 5.0, drift: float = 0.5):
    values = series.astype(float).to_numpy()
    mean = float(np.mean(values))
    std = float(np.std(values))
    if std == 0:
        std = 1.0

    normalized = (values - mean) / std
    cusum_pos = np.zeros(len(values))
    cusum_neg = np.zeros(len(values))
    change_points = []

    for i in range(1, len(values)):
        cusum_pos[i] = max(0.0, cusum_pos[i - 1] + normalized[i] - drift)
        cusum_neg[i] = max(0.0, cusum_neg[i - 1] - normalized[i] - drift)
        if cusum_pos[i] > threshold or cusum_neg[i] > threshold:
            change_points.append(i)
            cusum_pos[i] = 0.0
            cusum_neg[i] = 0.0

    return change_points, cusum_pos, cusum_neg


def prepare_series(frame: pd.DataFrame, column: str) -> pd.DataFrame:
    return frame[["date", column]].dropna().reset_index(drop=True)


def summarize_breaks(frame: pd.DataFrame, column: str, label: str, threshold: float = 5.0, drift: float = 0.5):
    series_frame = prepare_series(frame, column)
    change_points, cusum_pos, cusum_neg = cusum_detect(series_frame[column], threshold=threshold, drift=drift)
    breaks = pd.DataFrame(
        {
            "series": label,
            "column": column,
            "break_index": change_points,
            "break_date": [series_frame.loc[idx, "date"] for idx in change_points],
            "series_value": [series_frame.loc[idx, column] for idx in change_points],
            "threshold": threshold,
            "drift": drift,
        }
    )
    return series_frame, breaks, cusum_pos, cusum_neg


def plot_series_with_breaks(series_frame: pd.DataFrame, breaks: pd.DataFrame, column: str, label: str):
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=series_frame["date"],
            y=series_frame[column],
            mode="lines",
            name=label,
            line={"width": 2.5, "color": "#c46f3b"},
        )
    )
    for _, row in breaks.iterrows():
        break_date = pd.Timestamp(row["break_date"]).to_pydatetime()
        fig.add_shape(
            type="line",
            x0=break_date,
            x1=break_date,
            y0=0,
            y1=1,
            xref="x",
            yref="paper",
            line={"dash": "dash", "color": "#356c86"},
        )
        # fig.add_annotation(
        #     x=break_date,
        #     y=1,
        #     xref="x",
        #     yref="paper",
        #     text=break_date.strftime("%b %Y"),
        #     showarrow=False,
        #     yshift=12,
        # )
    fig.update_layout(
        title=f"CUSUM break detection: {label}",
        xaxis_title="Date",
        yaxis_title=label,
        template="plotly_white",
        height=420,
    )
    return fig


def plot_cusum_stats(series_frame: pd.DataFrame, cusum_pos: np.ndarray, cusum_neg: np.ndarray, label: str, threshold: float):
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=series_frame["date"], y=cusum_pos, mode="lines", name="S+", line={"color": "#c46f3b"}))
    fig.add_trace(go.Scatter(x=series_frame["date"], y=cusum_neg, mode="lines", name="S-", line={"color": "#356c86"}))
    fig.add_hline(y=threshold, line_dash="dot", line_color="#a34831")
    fig.update_layout(
        title=f"CUSUM statistics: {label}",
        xaxis_title="Date",
        yaxis_title="Cumulative sum (std-dev units)",
        template="plotly_white",
        height=420,
    )
    return fig

In [45]:
threshold = 5.0
drift = 0.5

break_series_map = {
    "wti_mom_pct": "WTI MoM Change (%)",
    "rig_count_diff": "Rig Count Change",
    "us_production_mbbld_diff": "U.S. Production Change (mbbl/d)",
}

cusum_results = {}
break_tables = []

for column, label in break_series_map.items():
    frame, breaks, cusum_pos, cusum_neg = summarize_breaks(
        monthly,
        column,
        label,
        threshold=threshold,
        drift=drift,
    )
    cusum_results[column] = {
        "label": label,
        "frame": frame,
        "breaks": breaks,
        "cusum_pos": cusum_pos,
        "cusum_neg": cusum_neg,
    }
    break_tables.append(breaks)

break_summary = pd.concat(break_tables, ignore_index=True)
break_summary

,series,column,break_index,break_date,series_value,threshold,drift
0,WTI MoM Change (%),wti_mom_pct,87,2020-04-01,-42.807797,5.0,0.5
1,WTI MoM Change (%),wti_mom_pct,89,2020-06-01,43.722716,5.0,0.5
2,Rig Count Change,rig_count_diff,24,2015-02-01,-347.500000,5.0,0.5
3,Rig Count Change,rig_count_diff,26,2015-04-01,-238.500000,5.0,0.5
4,Rig Count Change,rig_count_diff,87,2020-05-01,-226.600000,5.0,0.5
5,U.S. Production Change (mbbl/d),us_production_mbbld_diff,87,2020-05-01,-2195.000000,5.0,0.5


In [46]:
plot_series_with_breaks(
    cusum_results["wti_mom_pct"]["frame"],
    cusum_results["wti_mom_pct"]["breaks"],
    "wti_mom_pct",
    cusum_results["wti_mom_pct"]["label"],
)

In [23]:
plot_cusum_stats(
    cusum_results["wti_mom_pct"]["frame"],
    cusum_results["wti_mom_pct"]["cusum_pos"],
    cusum_results["wti_mom_pct"]["cusum_neg"],
    cusum_results["wti_mom_pct"]["label"],
    threshold,
)

The break summary identifies major regime shifts, especially around 2015 for drilling and 2020 for all three transformed series. These break periods are relevant because the oil-price response relationship may not be stable across the full sample.

## Part 2: Lead-Lag Analysis on Monthly National Levels

This section quantifies the monthly lead-lag relationship between WTI and the two national activity series.

### Inputs to the lead-lag section

- source series: `wti_price_weekly` at monthly frequency
- target series: `rig_count` and `us_production_mbbld`

### Methods

Two complementary estimators are used:

- Cross-correlation function (CCF): measures the direction and strength of linear association across positive and negative lags
- Lagged OLS regression: estimates `target_(t+k) ~ WTI_t` for each nonnegative lag and reports effect size and `R^2`

This combination is intended to show both the most likely lead-lag timing and the amount of variation explained at that lag.

In [24]:
def compute_cross_correlation(source: pd.Series, target: pd.Series, max_lag: int = 18) -> pd.DataFrame:
    data = pd.DataFrame({"source": source, "target": target}).dropna().reset_index(drop=True)
    rows = []
    for lag in range(-max_lag, max_lag + 1):
        if lag >= 0:
            paired = pd.DataFrame({"x": data["source"], "y": data["target"].shift(-lag)}).dropna()
        else:
            paired = pd.DataFrame({"x": data["source"].shift(lag), "y": data["target"]}).dropna()
        ccf = paired["x"].corr(paired["y"]) if len(paired) > 2 else np.nan
        rows.append({"lag": lag, "ccf": ccf, "n": len(paired)})

    ccf_df = pd.DataFrame(rows)
    sample_n = int(data.shape[0])
    ccf_df["ci_95"] = 1.96 / np.sqrt(sample_n) if sample_n else np.nan
    return ccf_df


def compute_lagged_regression(source: pd.Series, target: pd.Series, max_lag: int = 18) -> pd.DataFrame:
    data = pd.DataFrame({"source": source, "target": target}).dropna().reset_index(drop=True)
    rows = []
    for lag in range(0, max_lag + 1):
        paired = pd.DataFrame({"x": data["source"], "y": data["target"].shift(-lag)}).dropna()
        if len(paired) < 3:
            continue

        x = paired["x"].to_numpy(dtype=float)
        y = paired["y"].to_numpy(dtype=float)
        x_mean = x.mean()
        y_mean = y.mean()
        ss_xx = np.sum((x - x_mean) ** 2)
        ss_yy = np.sum((y - y_mean) ** 2)
        ss_xy = np.sum((x - x_mean) * (y - y_mean))
        beta = ss_xy / ss_xx if ss_xx else np.nan
        alpha = y_mean - beta * x_mean if ss_xx else np.nan
        fitted = alpha + beta * x
        resid = y - fitted
        sse = np.sum(resid ** 2)
        stderr = np.sqrt((sse / (len(x) - 2)) / ss_xx) if len(x) > 2 and ss_xx else np.nan
        corr = ss_xy / np.sqrt(ss_xx * ss_yy) if ss_xx and ss_yy else np.nan

        rows.append(
            {
                "lag": lag,
                "beta": beta,
                "std_err": stderr,
                "t_stat": beta / stderr if stderr and not np.isnan(stderr) else np.nan,
                "r_squared": corr ** 2 if not np.isnan(corr) else np.nan,
                "n": len(paired),
            }
        )
    return pd.DataFrame(rows)


def plot_ccf(ccf_df: pd.DataFrame, label: str):
    colors = ["#c46f3b" if abs(v) > c else "#b9b2a5" for v, c in zip(ccf_df["ccf"], ccf_df["ci_95"])]
    fig = go.Figure()
    fig.add_trace(go.Bar(x=ccf_df["lag"], y=ccf_df["ccf"], marker_color=colors, name="CCF"))
    ci = float(ccf_df["ci_95"].iloc[0])
    fig.add_hline(y=ci, line_dash="dot", line_color="#a34831")
    fig.add_hline(y=-ci, line_dash="dot", line_color="#a34831")
    fig.add_vline(x=0, line_dash="dash", line_color="#356c86")
    fig.update_layout(
        title=f"Cross-correlation: WTI vs {label}",
        xaxis_title="Lag in months (positive = WTI leads target)",
        yaxis_title="CCF",
        template="plotly_white",
        height=420,
    )
    return fig


def plot_lagged_regression(reg_df: pd.DataFrame, label: str):
    fig = go.Figure()
    fig.add_trace(go.Bar(x=reg_df["lag"], y=reg_df["r_squared"], name="R-squared", marker_color="#d5b45f"))
    fig.add_trace(
        go.Scatter(
            x=reg_df["lag"],
            y=reg_df["beta"],
            mode="lines+markers",
            name="Beta",
            yaxis="y2",
            line={"color": "#356c86", "width": 2},
        )
    )
    fig.update_layout(
        title=f"Lagged regression: WTI vs {label}",
        xaxis_title="Lag in months (target at t+k)",
        yaxis={"title": "R-squared"},
        yaxis2={"title": "Beta", "overlaying": "y", "side": "right"},
        template="plotly_white",
        height=420,
    )
    return fig

In [25]:
lead_lag_targets = {
    "rig_count": "National Rig Count",
    "us_production_mbbld": "U.S. Production (mbbl/d)",
}

lead_lag_results = {}
for column, label in lead_lag_targets.items():
    lead_lag_results[column] = {
        "label": label,
        "ccf": compute_cross_correlation(monthly["wti_price_weekly"], monthly[column], max_lag=18),
        "lag_reg": compute_lagged_regression(monthly["wti_price_weekly"], monthly[column], max_lag=18),
    }

summary_rows = []
for column, result in lead_lag_results.items():
    ccf_df = result["ccf"]
    reg_df = result["lag_reg"]
    peak_ccf = ccf_df.iloc[ccf_df["ccf"].abs().idxmax()]
    best_reg = reg_df.iloc[reg_df["r_squared"].idxmax()]
    summary_rows.append(
        {
            "target": result["label"],
            "peak_ccf_lag": int(peak_ccf["lag"]),
            "peak_ccf": float(peak_ccf["ccf"]),
            "ccf_significant": abs(float(peak_ccf["ccf"])) > float(peak_ccf["ci_95"]),
            "best_reg_lag": int(best_reg["lag"]),
            "best_beta": float(best_reg["beta"]),
            "best_r_squared": float(best_reg["r_squared"]),
            "best_t_stat": float(best_reg["t_stat"]) if not np.isnan(best_reg["t_stat"]) else np.nan,
        }
    )

lead_lag_summary = pd.DataFrame(summary_rows)
lead_lag_summary

,target,peak_ccf_lag,peak_ccf,ccf_significant,best_reg_lag,best_beta,best_r_squared,best_t_stat
0,National Rig Count,4,0.646320,True,4,15.919577,0.417729,10.338995
1,U.S. Production (mbbl/d),-18,0.209007,True,16,10.719576,0.019496,1.650463


## Summary Fields

The summary table reports one peak correlation result and one peak regression result for each target series.

- `peak_ccf_lag`: lag with the largest absolute cross-correlation
- `peak_ccf`: correlation at that lag
- `ccf_significant`: whether the peak absolute correlation exceeds the simple 95% confidence band
- `best_reg_lag`: lag with the largest regression `R^2`
- `best_beta`: slope coefficient from `target_(t+k) ~ WTI_t`
- `best_r_squared`: fraction of target variance explained at that lag
- `best_t_stat`: standardized slope estimate for that lagged regression

The key interpretation points are the sign and size of the best lag, and whether the associated correlation and regression fit are economically meaningful.

In [26]:
plot_ccf(lead_lag_results["rig_count"]["ccf"], lead_lag_results["rig_count"]["label"])

In [27]:
plot_lagged_regression(lead_lag_results["rig_count"]["lag_reg"], lead_lag_results["rig_count"]["label"])

In [28]:
plot_ccf(lead_lag_results["us_production_mbbld"]["ccf"], lead_lag_results["us_production_mbbld"]["label"])

In [29]:
plot_lagged_regression(lead_lag_results["us_production_mbbld"]["lag_reg"], lead_lag_results["us_production_mbbld"]["label"])

## Full-Sample Interpretation

### WTI vs National Rig Count

Observed pattern in the current project data:

- strongest CCF at about `+4` months
- peak CCF around `0.60`
- best lagged-regression `R^2` around `0.42`
- positive beta at all tested lags, with the strongest fit near 4 months

Interpretation: monthly WTI tends to lead national rig count by about 4 months, and the relationship is economically meaningful.

### WTI vs U.S. Production

Observed pattern in the current project data:

- peak CCF is small relative to the rig-count relationship
- best lagged-regression `R^2` is close to zero
- the strongest full-sample production lag has very low explanatory power

Interpretation: the full-sample monthly relationship between WTI and aggregate U.S. production is weak. Production appears slower-moving, more regime-dependent, and less directly tied to a single-lag WTI signal than rig count.

## Limitations

- The lead-lag estimates are full-sample summaries and may mask regime changes.
- The lagged OLS models are simple one-predictor models and do not control for macroeconomic variables.
- CUSUM identifies break timing but does not, by itself, estimate causal effect size.
- Aggregate U.S. production can respond with longer and more distributed lags than a single-lag model can capture.

## Summary

This notebook documents the monthly national-series evidence for oil-price transmission.

- National rig count exhibits a strong lead-lag relationship with WTI, with the strongest response around 4 months after price movements.
- Aggregate U.S. production shows a much weaker full-sample monthly relationship with WTI.
- CUSUM break detection highlights important regime changes, especially in 2015 and 2020, which motivates future regime-specific lag modeling.

## Recommended Next Step

The next analytical extension should estimate the same CCF and lagged-regression metrics separately across pre-break and post-break regimes so the timing and strength of oil-price transmission can be compared across structural periods.